In [1]:
import os
import pandas as pd
import numpy as np

In [3]:
RAW_DIR = "../data/raw"
PROCESSED_DIR = "../data/processed"

os.makedirs(PROCESSED_DIR, exist_ok=True)

usgs = pd.read_csv(os.path.join(RAW_DIR, "usgs_streamflow.csv"), parse_dates=["timestamp"])
rain = pd.read_csv(os.path.join(RAW_DIR, "rainfall_daily.csv"), parse_dates=["timestamp"])
flow = pd.read_csv(os.path.join(RAW_DIR, "interceptor_flow.csv"), parse_dates=["timestamp"])

In [4]:
usgs = usgs.sort_values("timestamp").set_index("timestamp")
rain = rain.sort_values("timestamp").set_index("timestamp")
flow = flow.sort_values("timestamp").set_index("timestamp")

In [5]:
usgs["streamflow_cfs"] = usgs["streamflow_cfs"].interpolate(limit=3)
flow["interceptor_flow_gpm"] = flow["interceptor_flow_gpm"].interpolate(limit=3)

In [6]:
rain["rainfall_in"] = rain["rainfall_in"].fillna(0)

In [7]:
flow["flow_mgd"] = flow["interceptor_flow_gpm"] * 1440 / 1_000_000
flow = flow.drop(columns=["interceptor_flow_gpm"])

In [8]:
df = flow.join(usgs, how="left").join(rain, how="left")
df.head()

,flow_mgd,streamflow_cfs,rainfall_in
timestamp,,,
2022-01-01,2173.493634,3370,0.04
2022-01-02,3439.352663,5320,0.03
2022-01-03,4761.824749,7380,0.47
2022-01-04,8925.863897,13800,0.05
2022-01-05,8783.973851,13600,0.85


In [9]:
DESIGN_CAPACITY_MGD = 120

df["capacity_pct"] = (df["flow_mgd"] / DESIGN_CAPACITY_MGD) * 100

In [10]:
df.isna().sum()
df.describe()

,flow_mgd,streamflow_cfs,rainfall_in,capacity_pct
count,731.000000,731.000000,731.000000,731.000000
mean,4725.173110,7310.603283,0.229508,3937.644259
std,5420.842656,8387.585332,0.307162,4517.368880
min,277.894683,450.000000,0.000000,231.578902
25%,1598.822996,2460.000000,0.030000,1332.352497
50%,3302.908499,5110.000000,0.110000,2752.423749
75%,6146.425684,9510.000000,0.310000,5122.021404
max,65929.134614,102000.000000,3.160000,54940.945512
